# Transfer Learning for SteelBlast Image Classification

Using a pre-trained ResNet50 neural network to classify steel surfaces as "good" or "not-good" for shot-blasting quality assessment.

## 1. Import Required Libraries

In [ ]:
from __future__ import annotations

import os
import json
import sys
import tempfile
from pathlib import Path
from dataclasses import dataclass

# Set matplotlib backend before importing
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold
import seaborn as sns
from skimage import exposure

import tensorflow as tf
from tensorflow import keras

# Reproducibility seed
RANDOM_SEED = 42
os.environ.setdefault('PYTHONHASHSEED', str(RANDOM_SEED))
np.random.seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

## Pipeline Configuration

This cell initializes runtime and pipeline settings before dataset loading and model training.

#### Approach
- Detect whether execution is in Colab (`detected_in_colab`) to support portable behavior.
- Define the main pipeline configuration through `PipelineConfig`.
- Apply local-only environment settings when `local_env=True`.
- Detect available GPUs and enforce the user policy from `enable_gpu`.
- Optionally enable mixed precision when GPU execution is active.
- Print the resolved runtime state for traceability.

#### Parameters
##### `PipelineConfig`
- `dataset_dir: Path`
  - Root folder containing train/test class subfolders.
- `model_choice: str`
  - Backbone selector fixed to `ResNet50`.
- `in_colab: bool`
  - Indicates whether code is running in Google Colab.
- `local_env: bool`
  - Enables local-only environment settings (for example Apple Silicon tuning).
- `enable_gpu: bool`
  - User-controlled flag to allow GPU execution.
- `use_gpu: bool`
  - Effective GPU status after availability checks and policy application.


In [ ]:
detected_in_colab = "google.colab" in sys.modules
print(f"Running in Colab: {detected_in_colab}")
print(f"TensorFlow version: {tf.__version__}")


@dataclass(frozen=True)
class PipelineConfig:
    """Dataset path and runtime environment flags."""
    dataset_dir: Path = Path("data/SteelBlastQC")
    model_choice: str = 'ResNet50'
    in_colab: bool = False
    local_env: bool = True # Set True to apply Apple-Silicon local env vars
    enable_gpu: bool = True # User-controlled: set True to request GPU use
    enable_mixed_precision: bool = False # Safer default on Apple Metal; enable only if stable
    reproducibility_mode: str = 'practical' # practical | strict
    deterministic_tf_ops: bool = True
    deterministic_data_pipeline: bool = True
    deterministic_shuffle: bool = True
    reshuffle_each_iteration: bool = False
    use_gpu: bool = False # Effective value resolved at runtime


LABELS = {"good": 0, "not-good": 1}
CLASS_NAMES = ["good", "not-good"]

pipeline_config = PipelineConfig(in_colab=detected_in_colab, local_env=not detected_in_colab)
reproducibility_mode = str(pipeline_config.reproducibility_mode).lower()
if reproducibility_mode not in {'practical', 'strict'}:
    raise ValueError(f"Unsupported reproducibility_mode: {pipeline_config.reproducibility_mode}")

if pipeline_config.deterministic_tf_ops:
    try:
        tf.config.experimental.enable_op_determinism()
        print("TensorFlow op determinism enabled")
    except Exception as e:
        print(f"Could not enable TensorFlow op determinism: {e}")


# Execute local-only environment variables only when local_env is enabled.
if pipeline_config.local_env:
    os.environ.setdefault("TF_GPU_ALLOCATOR", "metal")
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    print("Applied local environment variables (TF_GPU_ALLOCATOR, TF_CPP_MIN_LOG_LEVEL)")
else:
    print("Skipped local environment variables because local_env=False")

# Detect and optionally enable GPU based on config flag
detected_gpus = tf.config.list_physical_devices('GPU')
gpu_available = len(detected_gpus) > 0

if gpu_available and not pipeline_config.enable_gpu:
    # Disable GPU visibility so TensorFlow runs on CPU when the flag is off.
    try:
        tf.config.set_visible_devices([], 'GPU')
        detected_gpus = []
        gpu_available = False
        print("GPU disabled by AppConfig.enable_gpu=False")
    except RuntimeError as e:
        print("Could not disable GPU after initialization:", e)

use_gpu = pipeline_config.enable_gpu and gpu_available

if use_gpu:
    try:
        for gpu in detected_gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.config.optimizer.set_jit(False)
        print("Enabled memory growth for GPU; XLA JIT disabled")
    except RuntimeError as e:
        print("Could not set GPU memory growth:", e)

if use_gpu and pipeline_config.enable_mixed_precision:
    policy = tf.keras.mixed_precision.Policy("mixed_float16")
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f"Enabled mixed precision policy: {policy.name}")
else:
    tf.keras.mixed_precision.set_global_policy("float32")
    print("Mixed precision disabled (safer default for this pipeline)")

print("JIT enabled after config:", tf.config.optimizer.get_jit())
print(f"Model: {pipeline_config.model_choice}")
print(f"Dataset directory: {pipeline_config.dataset_dir}")
print(f"In Colab: {pipeline_config.in_colab}")
print(f"Local env flag: {pipeline_config.local_env}")
print(f"GPU available: {gpu_available}")
print(f"GPU enabled by flag: {pipeline_config.enable_gpu}")
print(f"Using GPU: {use_gpu}")
print(f"Mixed precision enabled: {pipeline_config.enable_mixed_precision and use_gpu}")
print(f"Reproducibility mode: {reproducibility_mode}")
print(f"Deterministic TF ops: {pipeline_config.deterministic_tf_ops}")
print(f"Deterministic data pipeline: {pipeline_config.deterministic_data_pipeline}")
print(f"Deterministic shuffle: {pipeline_config.deterministic_shuffle}")
print(f"Reshuffle each epoch: {pipeline_config.reshuffle_each_iteration}")


## Load and Pre-process Dataset

This cell defines preprocessing settings, loads dataset splits from disk, applies ResNet50 preprocessing, and builds efficient `tf.data` pipelines for training, validation, and testing.

#### Approach
- Define image and split settings with `PreprocessConfig`.
- Use a fixed ResNet50 input size of `(224, 224)`.
- Load image paths and class labels from the dataset folder structure.
- Create a stratified train/validation split to preserve class balance.
- Apply ResNet50 `preprocess_input` consistently across splits.
- Decode and resize images first, then apply training-only augmentation before model-specific preprocessing.
- Apply optional lighting hardening (probabilistic CLAHE) on the training branch only.
- Batch, cache, and prefetch with `tf.data` for efficient throughput.



#### Parameters
##### `PreprocessConfig`
- `image_size: tuple`
  - Input resolution for resizing images before preprocessing.
  - Fixed to `(224, 224)` for `ResNet50`.
- `batch_size: int`
  - Number of samples per batch.
- `validation_split: float`
  - Fraction of training data reserved for validation.
- `use_lighting_hardening: bool`
  - Enables lighting-robustness augmentation in training.
- `lighting_hardening_probability: float`
  - Probability of applying CLAHE to a training sample.
- `clahe_clip_limit: float`
  - CLAHE clipping strength for training-time normalization.

In [ ]:
@dataclass(frozen=True)
class PreprocessConfig:
    """Image size, batching, split, and lighting-hardening parameters."""
    image_size: tuple = (224, 224)
    batch_size: int = 32
    validation_split: float = 0.2
    use_lighting_hardening: bool = True
    lighting_hardening_probability: float = 0.30
    clahe_clip_limit: float = 0.01

preprocess_config = PreprocessConfig(batch_size=32, validation_split=0.2)

print(f"Image size for model {pipeline_config.model_choice}: {preprocess_config.image_size}")
print(f"Batch size: {preprocess_config.batch_size}")
print(f"Validation split: {preprocess_config.validation_split}")
print(f"Lighting hardening enabled: {preprocess_config.use_lighting_hardening}")
print(f"Lighting hardening probability: {preprocess_config.lighting_hardening_probability}")

import time
preprocessing_stage_start = time.perf_counter()

def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    """Load image paths and labels for train or test split."""
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)

# Load dataset splits
train_paths, y_train = load_split(pipeline_config.dataset_dir, "train")
test_paths, y_test = load_split(pipeline_config.dataset_dir, "test")

print(f"Train set: {len(train_paths)} images")
print(f"  - Good: {(y_train == 0).sum()}")
print(f"  - Not-good: {(y_train == 1).sum()}")
print(f"\nTest set: {len(test_paths)} images")
print(f"  - Good: {(y_test == 0).sum()}")
print(f"  - Not-good: {(y_test == 1).sum()}")

# Split train into train and validation

train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths, y_train, test_size=preprocess_config.validation_split, 
    random_state=42, stratify=y_train
)

print(f"\nAfter train/val split:")
print(f"Train: {len(train_paths)}, Validation: {len(val_paths)}, Test: {len(test_paths)}")

preprocess_fn = resnet_preprocess

print("Preparing tf.data datasets with ResNet50 preprocessing...")

def decode_image(image_path, label, image_size):
    """Decode and resize only; model-specific preprocessing is applied later."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32)
    return image, label

def _apply_clahe_numpy(image_np: np.ndarray, clip_limit: float) -> np.ndarray:
    image_np = np.clip(image_np / 255.0, 0.0, 1.0).astype(np.float32)
    normalized_channels = [
        exposure.equalize_adapthist(image_np[..., channel], clip_limit=float(clip_limit))
        for channel in range(image_np.shape[-1])
    ]
    normalized = np.stack(normalized_channels, axis=-1)
    return (np.clip(normalized, 0.0, 1.0) * 255.0).astype(np.float32)

def _apply_clahe_tf(image: tf.Tensor, image_size: tuple[int, int]) -> tf.Tensor:
    image = tf.py_function(
        func=lambda x: _apply_clahe_numpy(x, preprocess_config.clahe_clip_limit),
        inp=[image],
        Tout=tf.float32,
    )
    image.set_shape((image_size[0], image_size[1], 3))
    return image

# Data augmentation function with optional lighting hardening.
def augment(image, label, use_lighting_hardening: bool, image_size: tuple[int, int]):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=25.0)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

    if use_lighting_hardening:
        should_apply_clahe = tf.random.uniform(()) < preprocess_config.lighting_hardening_probability
        image = tf.cond(
            should_apply_clahe,
            lambda: _apply_clahe_tf(image, image_size),
            lambda: image,
        )

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def preprocess_for_model(image, label):
    image = preprocess_fn(image)
    return image, label


def build_dataset(
    image_paths,
    labels,
    batch_size,
    is_training=False,
    use_lighting_hardening: bool | None = None,
    image_size_override: tuple[int, int] | None = None,
):
    image_paths = [str(path) for path in image_paths
    ]
    labels = labels.astype(np.int32)

    image_size = preprocess_config.image_size if image_size_override is None else tuple(image_size_override)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(
        lambda image_path, label: decode_image(image_path, label, image_size),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    if pipeline_config.deterministic_data_pipeline:
        options = tf.data.Options()
        options.experimental_deterministic = True
        dataset = dataset.with_options(options)

    if is_training:
        dataset = dataset.shuffle(
            buffer_size=min(len(image_paths), 1000),
            seed=RANDOM_SEED if pipeline_config.deterministic_shuffle else None,
            reshuffle_each_iteration=pipeline_config.reshuffle_each_iteration if pipeline_config.deterministic_shuffle else True,
        )
        hardening_flag = preprocess_config.use_lighting_hardening if use_lighting_hardening is None else use_lighting_hardening
        dataset = dataset.map(
            lambda image, label: augment(image, label, hardening_flag, image_size),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.map(preprocess_for_model, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)

    # Avoid caching augmented training data in-memory to prevent kernel OOM crashes.
    if not is_training:
        dataset = dataset.cache()

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = build_dataset(train_paths, y_train, preprocess_config.batch_size, is_training=True)
val_dataset = build_dataset(val_paths, y_val, preprocess_config.batch_size)
test_dataset = build_dataset(test_paths, y_test, preprocess_config.batch_size)
preprocessing_time_seconds = float(time.perf_counter() - preprocessing_stage_start)

print(f"Train dataset batches: {len(train_paths) // preprocess_config.batch_size} (+ remainder)")
print(f"Validation samples: {len(val_paths)}")
print(f"Test samples: {len(test_paths)}")
print(f"Preprocessing stage time (s): {preprocessing_time_seconds:.2f}")

## Transfer Learning

This cell defines transfer-learning settings and loads a pretrained `ResNet50` backbone with configurable freezing behavior.

#### Approach inspired from (Ruzavina et al., 2025)
- Define backbone-loading options in `TransferLearningConfig`.
- Build `ResNet50` with fixed configuration for this pipeline.
- Set `include_top=False` to remove the original ImageNet classifier head.
- Load pretrained weights from `imagenet` (or another configured source).
- Apply `trainable` to freeze or unfreeze backbone layers for feature extraction or fine-tuning.
- Print resolved settings and model statistics for traceability.

#### Parameters
##### `TransferLearningConfig`
- `include_top: bool`
  - Whether to keep the original classifier head from the pretrained model.
  - Typically `False` when adding a custom task-specific head.
- `weights: str`
  - Weight initialization source (for example `imagenet` or `None`).
- `trainable: bool`
  - `False` freezes the backbone (feature extraction mode).
  - `True` unfreezes the backbone (fine-tuning mode).


In [ ]:
@dataclass(frozen=True)
class TransferLearningConfig:
    """Configuration for loading and freezing the pretrained ResNet50 backbone."""
    include_top: bool = False
    weights: str = 'imagenet'
    trainable: bool = False
    
transfer_learning_config = TransferLearningConfig()
print(f"Transfer learning include_top: {transfer_learning_config.include_top}")
print(f"Transfer learning weights: {transfer_learning_config.weights}")
print(f"Transfer learning trainable: {transfer_learning_config.trainable}")

# Load pre-trained ResNet50 with configured transfer-learning parameters
print(f"Loading ResNet50 with {transfer_learning_config.weights} weights...")

base_model = ResNet50(
    input_shape=(*preprocess_config.image_size, 3),
    include_top=transfer_learning_config.include_top,
    weights=transfer_learning_config.weights
)

# Freeze or unfreeze base model layers based on transfer-learning config
base_model.trainable = transfer_learning_config.trainable
print(f"Base model loaded with {len(base_model.layers)} layers")
print(f"Base model trainable: {base_model.trainable}")
print(f"Trainable parameters in base model: {base_model.count_params():,}")


## Modelling and Validation


#### Model Selection
This cell builds the final binary classifier by attaching a custom dense head to the pretrained backbone, then compiles and trains it with callbacks for convergence control.
- Build a transfer model by attaching a dense head to a pretrained ResNet50 backbone.
- Convert spatial feature maps into a compact vector using global pooling. Lin et al., 2013, "Network In Network"
- Learn task-specific nonlinear representations with dense layers.
- Apply dropout regularization to reduce overfitting.Srivastava et al., 2014
- Output class probability for binary classification (`not-good` as class 1).
- Select optimizer dynamically from configuration (`adam` or `sgd`).
- Compile with `binary_crossentropy` and `accuracy`.

#### Training and Validation approach
This cell runs a two-stage training and validation workflow for ResNet50 transfer learning: full-size baseline training, K-fold challenger screening, and full-size confirmatory training for the top candidate.
- Train a **baseline** model on full-size images (224x224 for ResNet50) and record runtime/F1 as the reference.
- Run **K-fold screening** on challenger candidates (`c1`, `c2`, `c3`) against CV validation metrics.
- Practical reproducibility mode is enabled by default: deterministic TF ops/data order + seeded shuffle with no per-epoch reshuffle.
- Select the best challenger by higher mean validation accuracy (tie-break by lower validation loss).
- Optionally run ranking stability check (full-size proxy) when enabled.
- Run one **full-size confirmatory training** for the selected challenger.
- Challenger is accepted only if `challenger_test_f1 >= baseline_test_f1`.
- Otherwise baseline is retained.


##### CV-Tuned Hyperparameters
- `learning_rate`
- `dense_units` (tuple for the two dense layers)
- `dropout_rates` (tuple for the two dropout layers)


##### Best found candidate:
- `name` = `baseline`
- `learning_rate` = 0.001
- `dense_units` = (256, 128)
- `dropout_rates` = (0.5, 0.3)


#### Parameters
##### `ModellingConfig`
- `epochs: int`
  - Maximum epochs for baseline/confirmatory full training.
- `learning_rate: float`
  - Baseline learning rate; challengers override this with candidate values.
- `dense_activation: str`
  - Hidden-layer activation function in the classifier head.
- `classification_activation: str`
  - Output activation for binary classification (typically `sigmoid`).
- `optimizer_function: str`
  - Optimizer family (`adam` or `sgd`).
- `early_stopping_patience: int`
  - Early stopping patience for full training phases.
- `reduce_lr_patience: int`
  - Learning-rate plateau patience for full training phases.
- `cv_folds: int`
  - Number of folds used in challenger screening.
- `screening_epochs: int`
  - Epochs per fold during challenger screening.
- `screening_image_size_primary: tuple[int, int]`
  - Reduced image size used during screening.
- `run_ranking_stability_check: bool`
  - Enables full-size proxy check for reduced-size ranking stability.

In [ ]:
@dataclass(frozen=True)
class ModellingConfig:
    """Model architecture and optimization controls."""
    # Maximum number of epochs to train the model
    epochs: int = 7
    # Learning rate for the optimizer
    learning_rate: float = 0.001
    # Activation function for dense layers
    dense_activation: str = 'relu'
    # Activation function for the output layer in binary classification
    classification_activation: str = 'sigmoid'
    # Optimizer function to use
    optimizer_function: str = 'adam'
    # Patience for early stopping
    early_stopping_patience: int = 10
    # Patience for reducing learning rate on plateau
    reduce_lr_patience: int = 5
    # CV screening controls
    cv_folds: int = 2
    screening_epochs: int = 2
    # Screening-only image-size strategy
    screening_image_size_primary: tuple[int, int] = (192, 192)
    # Perform optional ranking stability check against full-size screening
    run_ranking_stability_check: bool = True


model_config = ModellingConfig()

import time
import ast
import inspect
import textwrap


def _estimate_cyclomatic_complexity(fn) -> int | None:
    """Estimate cyclomatic complexity with a lightweight AST decision-point count."""
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        tree = ast.parse(src)
    except Exception:
        return None

    complexity = 1
    for node in ast.walk(tree):
        if isinstance(node, (ast.If, ast.For, ast.AsyncFor, ast.While, ast.IfExp, ast.ExceptHandler, ast.Assert, ast.Match)):
            complexity += 1
        elif isinstance(node, ast.BoolOp):
            complexity += max(len(node.values) - 1, 0)
        elif isinstance(node, ast.comprehension):
            complexity += len(node.ifs)
    return complexity


def _build_optimizer(learning_rate: float):
    if model_config.optimizer_function.lower() == 'adam':
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if model_config.optimizer_function.lower() == 'sgd':
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    raise ValueError(f"Unsupported optimizer_function: {model_config.optimizer_function}")


def _build_backbone(input_size: tuple[int, int]):
    backbone = ResNet50(
        input_shape=(*input_size, 3),
        include_top=transfer_learning_config.include_top,
        weights=transfer_learning_config.weights,
    )
    backbone.trainable = transfer_learning_config.trainable
    return backbone


def _build_model(
    learning_rate: float,
    dense_units: tuple[int, int],
    dropout_rates: tuple[float, float],
    input_size: tuple[int, int],
):
    tf.keras.backend.clear_session()
    backbone = _build_backbone(input_size=input_size)
    candidate_model = models.Sequential([
        backbone,
        layers.GlobalAveragePooling2D(),
        layers.Dense(dense_units[0], activation=model_config.dense_activation, name='dense_1'),
        layers.Dropout(dropout_rates[0]),
        layers.Dense(dense_units[1], activation=model_config.dense_activation, name='dense_2'),
        layers.Dropout(dropout_rates[1]),
        layers.Dense(1, activation=model_config.classification_activation, dtype='float32', name='output'),
    ])
    candidate_model.compile(
        optimizer=_build_optimizer(learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return candidate_model


def _make_callbacks(tag: str, early_patience: int, reduce_patience: int, save_checkpoint: bool = False):
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=early_patience,
            restore_best_weights=True,
            verbose=0,
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=reduce_patience,
            min_lr=1e-7,
            verbose=0,
        ),
    ]

    if save_checkpoint:
        callbacks.append(
            ModelCheckpoint(
                f"steelblast_{pipeline_config.model_choice.lower()}_{tag}_best.h5",
                monitor='val_accuracy',
                save_best_only=True,
                verbose=0,
            )
        )

    return callbacks


def _predict_labels(dataset, trained_model):
    preds = trained_model.predict(dataset, verbose=0).ravel()
    return (preds >= 0.5).astype(int)


def _f1_on_dataset(dataset, trained_model):
    labels = np.concatenate([y for x, y in dataset], axis=0)
    pred_labels = _predict_labels(dataset, trained_model)
    return float(f1_score(labels, pred_labels, average='weighted'))

pipeline_cyclomatic_complexity = {
    'build_dataset': _estimate_cyclomatic_complexity(build_dataset),
    'augment': _estimate_cyclomatic_complexity(augment),
    '_build_optimizer': _estimate_cyclomatic_complexity(_build_optimizer),
    '_build_backbone': _estimate_cyclomatic_complexity(_build_backbone),
    '_build_model': _estimate_cyclomatic_complexity(_build_model),
    '_make_callbacks': _estimate_cyclomatic_complexity(_make_callbacks),
    '_predict_labels': _estimate_cyclomatic_complexity(_predict_labels),
    '_f1_on_dataset': _estimate_cyclomatic_complexity(_f1_on_dataset),
}
pipeline_cyclomatic_complexity['total'] = int(sum(v for v in pipeline_cyclomatic_complexity.values() if isinstance(v, int)))
print(f"Estimated pipeline cyclomatic complexity total: {pipeline_cyclomatic_complexity['total']}")


print("Running tuning with 2-fold challenger screening.")
print("CV screening uses smaller inputs and lighter augmentation for speed.")

# Only these hyperparameters vary.
candidate_space = [
    {
        'name': 'baseline',
        'learning_rate': float(model_config.learning_rate),
        'dense_units': (256, 128),
        'dropout_rates': (0.5, 0.3),
    },
    {
        'name': 'c1',
        'learning_rate': 3e-4,
        'dense_units': (192, 96),
        'dropout_rates': (0.4, 0.2),
    },
    {
        'name': 'c2',
        'learning_rate': 2e-3,
        'dense_units': (192, 96),
        'dropout_rates': (0.4, 0.2),
    },
    {
        'name': 'c3',
        'learning_rate': 1e-3,
        'dense_units': (128, 64),
        'dropout_rates': (0.3, 0.2),
    },
]

full_size = tuple(preprocess_config.image_size)

# Baseline full run (reference for non-regression gate).
baseline_cfg = candidate_space[0]
baseline_model = _build_model(
    learning_rate=baseline_cfg['learning_rate'],
    dense_units=baseline_cfg['dense_units'],
    dropout_rates=baseline_cfg['dropout_rates'],
    input_size=full_size,
 )

print("\nBaseline full run")
print(f"lr={baseline_cfg['learning_rate']}, dense={baseline_cfg['dense_units']}, dropout={baseline_cfg['dropout_rates']}, size={full_size}")

baseline_start = time.time()
baseline_history = baseline_model.fit(
    train_dataset,
    epochs=model_config.epochs,
    validation_data=val_dataset,
    callbacks=_make_callbacks(
        tag='baseline_full',
        early_patience=model_config.early_stopping_patience,
        reduce_patience=model_config.reduce_lr_patience,
        save_checkpoint=False,
    ),
    verbose=1,
 )
baseline_time = float(time.time() - baseline_start)
baseline_val_acc = float(max(baseline_history.history.get('val_accuracy', [float('-inf')])))
baseline_val_loss = float(min(baseline_history.history.get('val_loss', [float('inf')])))
baseline_test_f1 = _f1_on_dataset(test_dataset, baseline_model)

print(f"Baseline runtime: {baseline_time:.2f}s")
print(f"Baseline test F1: {baseline_test_f1:.4f}")

tune_start = time.time()

# Stage 1: low-epoch CV screening (challengers only).
challengers = candidate_space[1:]
cv_records = []
train_paths_arr = np.asarray(train_paths, dtype=object)
y_train_arr = np.asarray(y_train, dtype=np.int64)
skf = StratifiedKFold(n_splits=model_config.cv_folds, shuffle=True, random_state=RANDOM_SEED)
split_indices = list(skf.split(train_paths_arr, y_train_arr))

screening_image_size = tuple(model_config.screening_image_size_primary)
global_fold_times = []

for cfg in challengers:
    print("\nCV screening", cfg['name'], f"@ size={screening_image_size}")
    fold_stats = []
    fold_times = []
    candidate_cv_start = time.time()

    for fold_idx, (train_idx, val_idx) in enumerate(split_indices, start=1):
        fold_train_paths = train_paths_arr[train_idx].tolist()
        fold_val_paths = train_paths_arr[val_idx].tolist()
        fold_y_train = y_train_arr[train_idx]
        fold_y_val = y_train_arr[val_idx]

        fold_train_ds = build_dataset(
            fold_train_paths,
            fold_y_train,
            preprocess_config.batch_size,
            is_training=True,
            use_lighting_hardening=False,
            image_size_override=screening_image_size,
        )
        fold_val_ds = build_dataset(
            fold_val_paths,
            fold_y_val,
            preprocess_config.batch_size,
            is_training=False,
            image_size_override=screening_image_size,
        )

        fold_model = _build_model(
            learning_rate=cfg['learning_rate'],
            dense_units=cfg['dense_units'],
            dropout_rates=cfg['dropout_rates'],
            input_size=screening_image_size,
        )

        fold_start = time.time()
        fold_history = fold_model.fit(
            fold_train_ds,
            epochs=min(model_config.screening_epochs, model_config.epochs),
            validation_data=fold_val_ds,
            callbacks=_make_callbacks(
                tag=f"{cfg['name']}_fold{fold_idx}",
                early_patience=1,
                reduce_patience=1,
                save_checkpoint=False,
            ),
            verbose=0,
        )
        fold_elapsed = float(time.time() - fold_start)
        fold_times.append(fold_elapsed)
        global_fold_times.append(fold_elapsed)

        fold_val_acc = float(max(fold_history.history.get('val_accuracy', [float('-inf')])))
        fold_val_loss = float(min(fold_history.history.get('val_loss', [float('inf')])))
        fold_stats.append({'fold': fold_idx, 'val_accuracy': fold_val_acc, 'val_loss': fold_val_loss, 'fold_time_seconds': fold_elapsed})

    if not fold_stats:
        continue

    cv_time = float(time.time() - candidate_cv_start)
    mean_val_acc = float(np.mean([f['val_accuracy'] for f in fold_stats]))
    mean_val_loss = float(np.mean([f['val_loss'] for f in fold_stats]))

    record = {
        'name': cfg['name'],
        'learning_rate': float(cfg['learning_rate']),
        'dense_units': list(cfg['dense_units']),
        'dropout_rates': list(cfg['dropout_rates']),
        'fold_stats': fold_stats,
        'mean_val_accuracy': mean_val_acc,
        'mean_val_loss': mean_val_loss,
        'cv_time_seconds': cv_time,
        'screening_use_lighting_hardening': False,
        'screening_save_checkpoint': False,
        'screening_image_size': list(screening_image_size),
    }
    cv_records.append(record)
    print(f"{cfg['name']} mean val acc={mean_val_acc:.4f}, mean val loss={mean_val_loss:.4f}, cv_time={cv_time:.2f}s")

# Pick top challenger by CV mean validation accuracy; tie-break lower validation loss.
best_cv = None
if cv_records:
    best_cv = sorted(
        cv_records,
        key=lambda r: (r['mean_val_accuracy'], -r['mean_val_loss']),
        reverse=True,
    )[0]

# Optional ranking stability check at full input size (single fold, single epoch).
ranking_stability_check = {
    'performed': False,
    'is_stable': None,
    'top_reduced': None,
    'top_full_proxy': None,
    'note': 'Not run.',
    'full_proxy_scores': [],
}

if model_config.run_ranking_stability_check and len(cv_records) >= 2:
    ranking_stability_check['performed'] = True
    reduced_sorted = sorted(cv_records, key=lambda r: (r['mean_val_accuracy'], -r['mean_val_loss']), reverse=True)
    top_two_names = [row['name'] for row in reduced_sorted[:2]]
    ranking_stability_check['top_reduced'] = top_two_names[0]

    ref_fold_train_idx, ref_fold_val_idx = split_indices[0]
    ref_train_paths = train_paths_arr[ref_fold_train_idx].tolist()
    ref_val_paths = train_paths_arr[ref_fold_val_idx].tolist()
    ref_y_train = y_train_arr[ref_fold_train_idx]
    ref_y_val = y_train_arr[ref_fold_val_idx]

    ref_train_ds = build_dataset(
        ref_train_paths,
        ref_y_train,
        preprocess_config.batch_size,
        is_training=True,
        use_lighting_hardening=False,
        image_size_override=full_size,
    )
    ref_val_ds = build_dataset(
        ref_val_paths,
        ref_y_val,
        preprocess_config.batch_size,
        is_training=False,
        image_size_override=full_size,
    )

    full_proxy_scores = []
    for candidate_name in top_two_names:
        cfg = next(item for item in challengers if item['name'] == candidate_name)
        proxy_model = _build_model(
            learning_rate=cfg['learning_rate'],
            dense_units=cfg['dense_units'],
            dropout_rates=cfg['dropout_rates'],
            input_size=full_size,
        )
        proxy_history = proxy_model.fit(
            ref_train_ds,
            epochs=1,
            validation_data=ref_val_ds,
            callbacks=_make_callbacks(
                tag=f"{candidate_name}_stability_proxy",
                early_patience=1,
                reduce_patience=1,
                save_checkpoint=False,
            ),
            verbose=0,
        )
        proxy_val_acc = float(max(proxy_history.history.get('val_accuracy', [float('-inf')])))
        proxy_val_loss = float(min(proxy_history.history.get('val_loss', [float('inf')])))
        full_proxy_scores.append({
            'name': candidate_name,
            'val_accuracy': proxy_val_acc,
            'val_loss': proxy_val_loss,
        })

    ranking_stability_check['full_proxy_scores'] = full_proxy_scores
    full_proxy_sorted = sorted(full_proxy_scores, key=lambda r: (r['val_accuracy'], -r['val_loss']), reverse=True)
    ranking_stability_check['top_full_proxy'] = full_proxy_sorted[0]['name']
    ranking_stability_check['is_stable'] = ranking_stability_check['top_reduced'] == ranking_stability_check['top_full_proxy']
    ranking_stability_check['note'] = "Stable" if ranking_stability_check['is_stable'] else "Unstable ranking between reduced and full-size proxy screening."

# Stage 2: one full confirmatory run for best challenger.
challenger_model = None
challenger_history = None
challenger_time = None
challenger_val_acc = None
challenger_val_loss = None
challenger_test_f1 = None
selected_cfg = baseline_cfg
selection_reason = "Baseline kept (challenger unavailable or no improvement)."

if best_cv is not None and ranking_stability_check['is_stable'] is False:
    best_cv = None
    selection_reason = "Baseline kept (reduced-size screening ranking unstable; use larger screening size)."

if best_cv is not None:
    cfg = next(item for item in challengers if item['name'] == best_cv['name'])
    print(f"\nConfirmatory full run for {cfg['name']}")
    challenger_model = _build_model(
        learning_rate=cfg['learning_rate'],
        dense_units=cfg['dense_units'],
        dropout_rates=cfg['dropout_rates'],
        input_size=full_size,
    )

    challenger_start = time.time()
    challenger_history = challenger_model.fit(
        train_dataset,
        epochs=model_config.epochs,
        validation_data=val_dataset,
        callbacks=_make_callbacks(
            tag=f"{cfg['name']}_full",
            early_patience=model_config.early_stopping_patience,
            reduce_patience=model_config.reduce_lr_patience,
            save_checkpoint=False,
        ),
        verbose=1,
    )
    challenger_time = float(time.time() - challenger_start)
    challenger_val_acc = float(max(challenger_history.history.get('val_accuracy', [float('-inf')])))
    challenger_val_loss = float(min(challenger_history.history.get('val_loss', [float('inf')])))
    challenger_test_f1 = _f1_on_dataset(test_dataset, challenger_model)

    # No-regression gate on test F1.
    if challenger_test_f1 >= baseline_test_f1:
        selected_cfg = cfg
        selection_reason = "Challenger selected (test F1 non-regression satisfied)."
    else:
        selection_reason = "Baseline kept (challenger test F1 lower than baseline)."

# Expose selected model/history to downstream evaluation cell.
if selected_cfg['name'] == baseline_cfg['name']:
    model = baseline_model
    history = baseline_history
    training_time_seconds = baseline_time
else:
    model = challenger_model
    history = challenger_history
    training_time_seconds = challenger_time

elapsed_tuning = float(time.time() - tune_start)

# Persist tuning summary.
tuning_summary = {
    'baseline': {
        'name': baseline_cfg['name'],
        'learning_rate': float(baseline_cfg['learning_rate']),
        'dense_units': list(baseline_cfg['dense_units']),
        'dropout_rates': list(baseline_cfg['dropout_rates']),
        'runtime_seconds': baseline_time,
        'best_val_accuracy': baseline_val_acc,
        'best_val_loss': baseline_val_loss,
        'test_f1': baseline_test_f1,
        'image_size': list(full_size),
    },
    'cv_screening': cv_records,
    'best_cv_candidate': None if best_cv is None else best_cv['name'],
    'confirmatory': None if challenger_model is None else {
        'name': best_cv['name'],
        'runtime_seconds': challenger_time,
        'best_val_accuracy': challenger_val_acc,
        'best_val_loss': challenger_val_loss,
        'test_f1': challenger_test_f1,
        'image_size': list(full_size),
    },
    'selected': {
        'name': selected_cfg['name'],
        'learning_rate': float(selected_cfg['learning_rate']),
        'dense_units': list(selected_cfg['dense_units']),
        'dropout_rates': list(selected_cfg['dropout_rates']),
    },
    'selection_reason': selection_reason,
    'ranking_stability_check': ranking_stability_check,
    'screening_image_size_primary': list(model_config.screening_image_size_primary),
    'reproducibility': {
        'seed': int(RANDOM_SEED),
        'mode': reproducibility_mode,
        'deterministic_tf_ops': bool(pipeline_config.deterministic_tf_ops),
        'deterministic_data_pipeline': bool(pipeline_config.deterministic_data_pipeline),
        'deterministic_shuffle': bool(pipeline_config.deterministic_shuffle),
        'reshuffle_each_iteration': bool(pipeline_config.reshuffle_each_iteration),
        'screening_epochs': int(model_config.screening_epochs),
    },
    'elapsed_tuning_seconds': elapsed_tuning,
}

print("\n" + "=" * 70)
print("TUNING SUMMARY")
print("=" * 70)
print(f"Selected candidate: {tuning_summary['selected']['name']}")
print(f"Selection reason: {selection_reason}")
print(f"Tuning elapsed: {elapsed_tuning:.2f}s")
print("=" * 70)


## Model Evaluation  

This cell evaluates the trained model on the held-out test split, generates classification diagnostics, and saves one merged run report for reproducibility.

#### Approach
- Evaluate final loss and accuracy on `test_dataset`.
- Convert sigmoid outputs to binary predictions using a `0.5` decision threshold.
- Compute detailed metrics (`accuracy`, `precision`, `recall`, `f1`).
- Generate a full classification report and confusion matrix.
- Save the trained model to disk using a model name derived from `pipeline_config.model_choice`.
- Merge tuning output and evaluation metrics into one timestamped JSON run report.
- Print a final summary block for quick comparison across model runs.



In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")

# Get predictions
# Use the model to predict on batches from the test dataset
import time
inference_start = time.perf_counter()
predictions = model.predict(test_dataset, verbose=0).ravel()
inference_time_seconds = float(time.perf_counter() - inference_start)
y_pred = (predictions >= 0.5).astype(int)

# Extract true labels from the dataset
true_labels = np.concatenate([y for x, y in test_dataset], axis=0)
inference_samples = int(len(true_labels))
inference_time_per_sample_ms = float((inference_time_seconds * 1000.0) / max(inference_samples, 1))
print(f"Inference time (s): {inference_time_seconds:.4f}")
print(f"Inference time per sample (ms): {inference_time_per_sample_ms:.4f}")

# Calculate detailed metrics
accuracy = accuracy_score(true_labels, y_pred)
precision = precision_score(true_labels, y_pred, average='weighted')
recall = recall_score(true_labels, y_pred, average='weighted')
f1 = f1_score(true_labels, y_pred, average='weighted')

print(f"\nDetailed Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Classification report
print(f"\nClassification Report:")
report_dict = classification_report(
    true_labels,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)
report_text = classification_report(
    true_labels,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)
print(report_text)

# Confusion Matrix
cm = confusion_matrix(true_labels, y_pred)
print(f"Confusion Matrix:\n{cm}")

# Save the trained model
model_path = f'steelblast_{pipeline_config.model_choice.lower()}.h5'
model.save(model_path)
print(f"Model saved to {model_path}")

# Build evaluation metrics payload
metrics = {
    "model": pipeline_config.model_choice,
    "image_size": preprocess_config.image_size,
    "classification_report": report_dict,
    "dataset_info": {
        "train_samples": len(train_paths),
        "val_samples": len(val_paths),
        "test_samples": len(test_paths),
        "classes": CLASS_NAMES
    },
    "confusion_matrix": cm.tolist(),
    "training_epochs": len(history.history['loss']),
    "training_time_seconds": training_time_seconds,
    "preprocessing_time_seconds": float(preprocessing_time_seconds) if 'preprocessing_time_seconds' in globals() else None,
    "inference_time_seconds": inference_time_seconds,
    "inference_samples": inference_samples,
    "inference_time_per_sample_ms": inference_time_per_sample_ms,
    "cyclomatic_complexity": pipeline_cyclomatic_complexity if 'pipeline_cyclomatic_complexity' in globals() else None,
    "batch_size": preprocess_config.batch_size
}

# Merge tuning and evaluation into one run report
merged_report = {
    "model": pipeline_config.model_choice,
    "tuning": tuning_summary,
    "evaluation": metrics,
    "reproducibility": tuning_summary.get("reproducibility", None)
}

from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
merged_report_path = f"steelblast_{pipeline_config.model_choice.lower()}_run_report_{timestamp}.json"
with open(merged_report_path, 'w') as f:
    json.dump(merged_report, f, indent=4)
print(f"Merged run report saved to {merged_report_path}")


# Display final summary
print("\n" + "="*60)
print(f"FINAL SUMMARY - {pipeline_config.model_choice}")
print("="*60)
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print("="*60)
